# Exercițiu Laborator 4 — Rețele Markov

## Enunțul problemei
Considerăm o rețea Markov (câmp aleator Markov — MRF) cu 5 variabile discrete binare $A_1, A_2, A_3, A_4, A_5 \in \{0, 1\}$. Muchiile grafului neorientat $G = (V, E)$ sunt:
* $A_1$ conectat cu $A_2$ și $A_3$: $(A_1, A_2), (A_1, A_3)$;
* $A_2$ conectat cu $A_4$ și $A_5$: $(A_2, A_4), (A_2, A_5)$;
* $A_3$ conectat cu $A_4$: $(A_3, A_4)$;
* $A_4$ conectat cu $A_5$: $(A_4, A_5)$.

Potențialele asociate clicilor maximale ale modelului sunt:
$$
\phi_C(a_C) = e^{\beta \sum_{i, j \in C, \, i < j} a_i a_j}.
$$

**Cerințe:**
* **a)** Vizualizați în `pyAgrum` graful rețelei Markov, iar apoi determinați clicile modelului.
* **b)** Determinați probabilitatea comună a celor 5 variabile și deduceți stările de probabilitate maximă pentru acestea.
* **c)** Explicați cum ați modifica funcțiile de potențial pentru a modela modelul Ising, în care sunt favorizate perechile de vecini care au aceeași valoare.

## a) Graful rețelei Markov și determinarea clicilor maximale

Mulțimea vârfurilor este $V = \{A_1, A_2, A_3, A_4, A_5\}$, iar mulțimea celor 6 muchii este:
$$
E = \{(A_1, A_2), (A_1, A_3), (A_2, A_4), (A_2, A_5), (A_3, A_4), (A_4, A_5)\}.
$$

Analizând subgrafurile complete:
* **Clici de mărime 1 (noduri):** $\{A_1\}, \{A_2\}, \{A_3\}, \{A_4\}, \{A_5\}$ (nu sunt maximale).
* **Clici de mărime 2 (muchii):** cele 6 muchii din $E$. Dintre acestea, muchiile $(A_2, A_4)$, $(A_2, A_5)$ și $(A_4, A_5)$ sunt incluse în triunghiul $\{A_2, A_4, A_5\}$, deci nu sunt maximale. Celelalte trei muchii nu fac parte din niciun triunghi:
  $$C_1 = \{A_1, A_2\}, \quad C_2 = \{A_1, A_3\}, \quad C_3 = \{A_3, A_4\}.$$
* **Clici de mărime 3 (triunghiuri):** există un singur triunghi complet:
  $$C_4 = \{A_2, A_4, A_5\}.$$
* **Clici de mărime $\ge 4$:** nu există (graful are 6 muchii, iar un $K_4$ necesită $\binom{4}{2} = 6$ muchii pe 4 vârfuri, însă nodurile $\{A_1, A_2, A_3, A_4\}$ formează un 4-ciclu fără coarde).

**Prin urmare, cele 4 clici maximale ale rețelei Markov sunt:**
$$
\mathcal{C}_{\max} = \Big\{ \{A_1, A_2\}, \; \{A_1, A_3\}, \; \{A_3, A_4\}, \; \{A_2, A_4, A_5\} \Big\}.
$$

In [ ]:
# Vizualizarea grafului în pyAgrum (cu fallback networkx/matplotlib dacă pyAgrum nu este instalat)
edges = [("A1", "A2"), ("A1", "A3"), ("A2", "A4"), ("A2", "A5"), ("A3", "A4"), ("A4", "A5")]

try:
    import pyAgrum as gum
    import pyAgrum.lib.notebook as gnb

    MRFClass = getattr(gum, "MarkovRandomField", getattr(gum, "MarkovNet", None))
    mn = MRFClass()
    for node in ["A1", "A2", "A3", "A4", "A5"]:
        mn.add(gum.LabelizedVariable(node, node, 2))
    for u, v in edges:
        mn.addEdge(u, v)

    print("Noduri:", [mn.variable(i).name() for i in mn.nodes()])
    print("Muchii:", [(mn.variable(u).name(), mn.variable(v).name()) for u, v in mn.edges()])
    gnb.showMRF(mn)
except ImportError:
    import networkx as nx
    import matplotlib.pyplot as plt

    print("pyAgrum nu este instalat în mediul curent; se afișează graful neorientat echivalent cu NetworkX:")
    G = nx.Graph()
    G.add_edges_from(edges)
    pos = {"A1": (0, 1), "A2": (1, 1.5), "A3": (1, 0.5), "A4": (2, 1), "A5": (2.8, 1.5)}
    plt.figure(figsize=(6.5, 4))
    nx.draw(G, pos, with_labels=True, node_color="#8ecae6", node_size=1400, font_weight="bold", edge_color="#023047", width=2)
    plt.title("Graful rețelei Markov (clica maximală de ordin 3: {A2, A4, A5})")
    plt.tight_layout()
    plt.show()

## b) Probabilitatea comună și stările de probabilitate maximă (MAP)

Conform **Teoremei Hammersley–Clifford**, distribuția comună a unei rețele Markov se factorizează ca produsul potențialelor peste clicile maximale, normalizat prin funcția de partiție $Z$:
$$
P(A_1=a_1, A_2=a_2, A_3=a_3, A_4=a_4, A_5=a_5) = \frac{1}{Z} \prod_{C \in \mathcal{C}_{\max}} \phi_C(a_C).
$$

Explicitând factorii pentru cele 4 clici maximale:
* $\phi_{\{A_1, A_2\}}(a_1, a_2) = e^{\beta a_1 a_2}$
* $\phi_{\{A_1, A_3\}}(a_1, a_3) = e^{\beta a_1 a_3}$
* $\phi_{\{A_3, A_4\}}(a_3, a_4) = e^{\beta a_3 a_4}$
* $\phi_{\{A_2, A_4, A_5\}}(a_2, a_4, a_5) = e^{\beta (a_2 a_4 + a_2 a_5 + a_4 a_5)}$

Observăm că fiecare muchie din $E$ apare într-o singură clică maximală, deci produsul potențialelor devine:
$$
\tilde{P}(a_1, a_2, a_3, a_4, a_5) = \exp\Big(\beta \big(a_1 a_2 + a_1 a_3 + a_3 a_4 + a_2 a_4 + a_2 a_5 + a_4 a_5\big)\Big) = \exp\left(\beta \sum_{(i, j) \in E} a_i a_j\right),
$$
iar constanta de normalizare (funcția de partiție) este:
$$
Z(\beta) = \sum_{(a_1, \dots, a_5) \in \{0, 1\}^5} \exp\left(\beta \sum_{(i, j) \in E} a_i a_j\right).
$$

### Stările de probabilitate maximă (MAP)
* **Cazul $\beta > 0$ (interacțiune atractivă, cazul standard):** Deoarece $a_i \in \{0, 1\}$, fiecare termen $a_i a_j \in \{0, 1\}$ este maxim (egal cu $1$) dacă și numai dacă $a_i = a_j = 1$. Suma $\sum_{(i,j)\in E} a_i a_j$ atinge maximul absolut $|E| = 6$ **unic** în configurația:
  $$(a_1^*, a_2^*, a_3^*, a_4^*, a_5^*) = (1, 1, 1, 1, 1),$$
  cu probabilitatea $P(1, 1, 1, 1, 1) = \frac{e^{6\beta}}{Z(\beta)}$ (pentru $\beta = 1$, $Z(1) \approx 706.2787$ și $P(1,1,1,1,1) = \frac{e^6}{Z(1)} \approx 0.5712$).
* **Cazul $\beta = 0$:** Toate cele $2^5 = 32$ stări sunt echiprobabile ($P(a) = \frac{1}{32}$).
* **Cazul $\beta < 0$ (interacțiune repulsivă):** Probabilitatea este maximă când $\sum_{(i,j)\in E} a_i a_j = 0$, adică pentru orice mulțime independentă a grafului (niciun cuplu de vecini nu are simultan valoarea $1$).

In [ ]:
import itertools
import numpy as np
import pandas as pd

beta = 1.0  # Parametru pozitiv de interacțiune
edge_idx = [(0, 1), (0, 2), (1, 3), (1, 4), (2, 3), (3, 4)]

records = []
for state in itertools.product([0, 1], repeat=5):
    active_edges = sum(state[u] * state[v] for u, v in edge_idx)
    unnorm_prob = np.exp(beta * active_edges)
    records.append({
        "A1": state[0], "A2": state[1], "A3": state[2], "A4": state[3], "A5": state[4],
        "Suma_muchii": active_edges,
        "Potential_prod": unnorm_prob
    })

df = pd.DataFrame(records)
Z = df["Potential_prod"].sum()
df["Probabilitate"] = df["Potential_prod"] / Z
df = df.sort_values(by="Probabilitate", ascending=False).reset_index(drop=True)

print(f"Constanta de normalizare Z(beta={beta}) = {Z:.6f}")
print("\nTop 10 stări după probabilitatea comună:")
print(df.head(10).to_string(index=False))

best_row = df.iloc[0]
print(f"\nStarea MAP unică (pentru beta > 0): ({int(best_row.A1)}, {int(best_row.A2)}, {int(best_row.A3)}, {int(best_row.A4)}, {int(best_row.A5)}) "
      f"cu P = {best_row.Probabilitate:.6f}")

## c) Modificarea funcțiilor de potențial pentru modelul Ising

În formularea inițială $\phi_C(a_C) = e^{\beta \sum_{i < j} a_i a_j}$ cu $a_i \in \{0, 1\}$, produsul $a_i a_j$ ia valoarea $1$ **doar** când $a_i = a_j = 1$, și valoarea $0$ atât când $(a_i, a_j) = (0, 0)$, cât și când $a_i \neq a_j$. Așadar, perechea $(1, 1)$ este favorizată în detrimentul perechii $(0, 0)$, deși în ambele cazuri vecinii au aceeași stare.

Pentru a modela **modelul Ising** (în care orice pereche de vecini cu **aceeași valoare** — fie $(0, 0)$, fie $(1, 1)$ — este favorizată simetric față de perechile cu valori opuse $(0, 1)$ și $(1, 0)$), putem modifica potențialele în oricare dintre următoarele două moduri echivalente (pentru $\beta > 0$):

1. **Prin maparea variabilelor la spini $s_i = 2a_i - 1 \in \{-1, +1\}$:**
   $$
   \phi_C^{\text{Ising}}(a_C) = \exp\left( \beta \sum_{i, j \in C, \, i < j} (2a_i - 1)(2a_j - 1) \right) = \exp\left( \beta \sum_{i, j \in C, \, i < j} s_i s_j \right).
   $$
   Aici, produsul $s_i s_j = +1$ ori de câte ori $a_i = a_j$ (contribuție $+\beta$ la exponent) și $s_i s_j = -1$ ori de câte ori $a_i \neq a_j$ (contribuție $-\beta$).

2. **Folosind funcția indicator de egalitate $\mathbb{I}(a_i = a_j) = 1 - |a_i - a_j| = 1 - (a_i - a_j)^2$:**
   $$
   \phi_C^{\text{Ising}}(a_C) = \exp\left( \beta \sum_{i, j \in C, \, i < j} \mathbb{I}(a_i = a_j) \right) = \exp\left( \beta \sum_{i, j \in C, \, i < j} \big(1 - (a_i - a_j)^2\big) \right).
   $$

Cele două expresii diferă doar printr-o constantă multiplicativă globală $e^{\beta |E|}$ care se reduce la normalizarea prin $Z$, definind exact aceeași distribuție de probabilitate. În acest model Ising feromagnetic ($\beta > 0$), există **două stări de probabilitate maximă (MAP) simetrice**: $(0, 0, 0, 0, 0)$ și $(1, 1, 1, 1, 1)$.